# 02 - Baseline RAG on the golden set

Dense retrieval (bge-m3 + exact FAISS index) followed by a 4-bit LLM that must answer from the retrieved
passages, cite them, and reply `NOT_FOUND` when they do not contain the answer. Everything runs on the
81 questions of `golden/golden_set.jsonl`.

Outputs: retrieval metrics (is the right chunk in the top-k?) and, per model, rule-based answer checks
(refusals, citation validity, number match). These are cheap proxies, not a judge: answers still need
reading, especially for `false_premise` questions.

Before running: Runtime > T4 GPU, Colab Secret `HF_TOKEN` (notebook access on). Each model downloads
roughly 16-19 GB and 81 questions take about 15-20 minutes per model.

In [ ]:
import os
import subprocess
import sys

REPO = "agentic-rag-eval"
BRANCH = "p1-agentic-rag"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random

import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

from src.eval.golden import load_golden, validate
from src.retrieval.embed import load_chunks

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/sec-10k-chunks"
chunks = load_chunks(hf_hub_download(HF_REPO, "chunks_1600.jsonl", repo_type="dataset"))
embeddings = np.load(
    hf_hub_download(HF_REPO, "embeddings_bge-m3_1600.npy", repo_type="dataset")
).astype(np.float32)
assert len(chunks) == len(embeddings), "chunks and embeddings are out of sync"
questions = load_golden("golden/golden_set.jsonl")
errors, _ = validate(questions, chunks)
assert not errors, errors
print(len(chunks), "chunks,", len(questions), "questions")

## Retrieval

Top-10 for every question; the generator later receives the top-5.

In [ ]:
import gc
import json

import pandas as pd
from sentence_transformers import SentenceTransformer

from src.eval.metrics import retrieval_metrics, summarise
from src.retrieval.embed import embed_texts
from src.retrieval.index import build_index, search

K = 5
index = build_index(embeddings)
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
embedder.max_seq_length = 640
query_vectors = embed_texts(embedder, [q["question"] for q in questions])
_, top_ids = search(index, query_vectors, k=10)
del embedder
gc.collect()
torch.cuda.empty_cache()

for q, ids in zip(questions, top_ids):
    q["retrieved_ids"] = [chunks[i]["chunk_id"] for i in ids]
    q["retrieved"] = [chunks[i] for i in ids[:K]]

rows = []
for q in questions:
    if q["expected"] == "answer":
        rows.append(
            {"id": q["id"], "type": q["type"]}
            | retrieval_metrics(q["evidence_chunk_ids"], q["retrieved_ids"])
        )
retrieval_summary = summarise(rows)
table = pd.DataFrame(retrieval_summary).T
print(table[["n", "hit@1", "hit@5", "hit@10", "recall@5", "all@5", "mrr"]].to_string())
os.makedirs("results", exist_ok=True)
with open("results/retrieval_metrics.json", "w") as f:
    json.dump(retrieval_summary, f, indent=1)

## Generation

Same retrieved context, same prompt, greedy decoding: only the model changes. Edit `MODELS` to change
the candidates.

In [ ]:
import time

from src.agent.llm import generate, load_llm
from src.agent.parsing import parse_answer
from src.agent.prompts import build_messages
from src.eval.metrics import score_row

MODELS = ["Qwen/Qwen3.5-9B", "Qwen/Qwen3-8B"]
all_rows = {}
for name in MODELS:
    print("=" * 80, "\nLoading", name)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    tokenizer, model, loader = load_llm(name)
    print(f"loaded with {loader} in {round(time.time() - t0)}s")
    records = []
    for q in questions:
        provided = [c["chunk_id"] for c in q["retrieved"]]
        out = generate(tokenizer, model, build_messages(q["question"], q["retrieved"]))
        parsed = parse_answer(out["text"], provided)
        row = score_row(q, parsed, q["retrieved_ids"])
        row |= {
            "model": name,
            "question": q["question"],
            "answer": parsed["answer"],
            "cited_ids": parsed["cited_ids"],
            "had_thinking": parsed["had_thinking"],
            "retrieved_top5": provided,
            "new_tokens": out["new_tokens"],
            "prompt_tokens": out["prompt_tokens"],
            "seconds": out["seconds"],
            "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1),
        }
        records.append(row)
        print(
            q["id"],
            q["type"],
            "refused" if row["refused"] else "answered",
            "ok" if row["behaviour_ok"] else "WRONG",
            f"{out['seconds']}s",
        )
    all_rows[name] = records
    short = name.split("/")[-1].lower()
    with open(f"results/baseline_{short}.jsonl", "w") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in records)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
pd.set_option("display.width", 200)
columns = [
    "n",
    "behaviour_ok",
    "refused",
    "invalid_citation",
    "cites_evidence",
    "uncited_answer",
    "numbers_match",
    "had_thinking",
    "seconds",
    "peak_vram_gb",
]
overall = {}
for name, records in all_rows.items():
    summary = summarise(records)
    print("\n" + name)
    print(pd.DataFrame(summary).T.reindex(columns=columns).to_string())
    overall[name] = summary
with open("results/baseline_summary.json", "w") as f:
    json.dump(overall, f, indent=1)

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("baseline_results", "zip", "results")
files.download("baseline_results.zip")

Copy back to the chat: the retrieval table, the summary table of each model and the "loaded with" lines.
The zip contains every answer so they can be read against the golden set.